# Summarising results

This notebook summarises results into a Pandas dataframe which is then
reformatted into a table suitable for publishing.

In [1]:
from qaoa_parameter_setting.utils.summary import (
    SummaryTable,
    formatted_styler_for,
    ACRONYM_MAPPING,
)
import pandas as pd
from datetime import datetime, timezone
from typing import Literal

In [2]:
# TABLE_JSON: str | None = "summary_tables.json"
TABLE_JSON: str | None = None
table = SummaryTable(TABLE_JSON, problem_classes=["MC"])

## Setup table with training data and min-max cuts.

In [3]:
if TABLE_JSON is None:
    # Add training data. Only the "best" results are kept, per graph, trainer config
    # file, and depth.
    table.add_data("../data/training/random_regular")
    table.add_data("../data/training/heavy_hex")
    table.add_data("../data/training/line_to_full")
    table.add_data("../data/training/erdos_renyi")

In [4]:
if TABLE_JSON is None:
    # Add min- and max-cut data. If some graph instances do not have a minmax_cuts
    # entry, an error will be thrown later.
    table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
    table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
    table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
    table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")

In [5]:
if TABLE_JSON is None:
    # Register all methods. This just keeps track of the methods so we can
    # identify missing training data.
    table.add_methods("../methods/")

## Identify missing min- and max-cuts data

In [6]:
missing_minmax_cuts = table.missing_minmax_cuts()
if len(missing_minmax_cuts) == 0:
    print("All minmax_cuts data accounted for.")
else:
    print(
        "The following graphs are missing min- and max-cuts data. "
        + "Generate them with compute_min_max_for_graph.py"
    )
    for _graph in missing_minmax_cuts:
        print("- {}".format(_graph))

All minmax_cuts data accounted for.


In [7]:
if TABLE_JSON is None:
    table.save_data("summary_tables.json", overwrite=True)

## Get raw data table

In [8]:
# This is the _raw_ table with all results
df: pd.DataFrame = table.to_dataframe()
df

,graph_type,graph_idx,num_nodes,trainer_config,method,depth,edge_probability,regular_degree,heavy_hex_rows,heavy_hex_cols,num_swap_layers,graph_key,energy,qaoa_angles,result_filename,trainer,evaluation,approximation_ratio
0,random_regular,10,10,FA_SV_noOpt.json,FA_noOpt.json,5,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,4.724871,"[0.6225358200399561, 0.5050671398134732, 0.416...",20250818_100912_000N10R4R_MC_FA_SV_noOpt_5.json,FixedAngleConjecture,SV,0.920304
1,random_regular,10,10,FA_SV_noOpt.json,FA_noOpt.json,1,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,2.185001,"[0.392757551327448, 0.5234801120857124]",20251130_132303_000N10R4R_MC_FA_SV_noOpt_1.json,FixedAngleConjecture,SV,0.761563
2,random_regular,10,10,FA_SV_noOpt.json,FA_noOpt.json,4,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,4.377104,"[0.6049882911890669, 0.4777982182658282, 0.361...",20251130_132306_000N10R4R_MC_FA_SV_noOpt_4.json,FixedAngleConjecture,SV,0.898569
3,random_regular,10,10,FA_SV_noOpt.json,FA_noOpt.json,2,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,3.096558,"[0.5343441634121997, 0.2830366020507894, 0.408...",20251130_132308_000N10R4R_MC_FA_SV_noOpt_2.json,FixedAngleConjecture,SV,0.818535
4,random_regular,10,10,FA_SV_noOpt.json,FA_noOpt.json,3,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,3.858886,"[0.5879466969458422, 0.423177520059303, 0.2230...",20251130_132316_000N10R4R_MC_FA_SV_noOpt_3.json,FixedAngleConjecture,SV,0.866180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72418,erdos_renyi,19,19,LR_SV_opt.json,LR_opt.json,10,0.5,NaN,NaN,NaN,NaN,001_19nodes_erdosrenyi50percent.json,13.302642,"[0.6392859788505468, 0.5753573809654922, 0.511...",20251210_103745_001N19ER50_MC_LR_SV_opt_10.json,TQATrainer,SV,0.938377
72419,erdos_renyi,20,20,LR_SV_opt.json,LR_opt.json,10,0.2,NaN,NaN,NaN,NaN,001_20nodes_erdosrenyi20percent.json,11.298735,"[0.6738254416166942, 0.6064428974550248, 0.539...",20251210_103831_001N20ER20_MC_LR_SV_opt_10.json,TQATrainer,SV,0.948447
72420,erdos_renyi,20,20,LR_SV_opt.json,LR_opt.json,10,0.3,NaN,NaN,NaN,NaN,001_20nodes_erdosrenyi30percent.json,12.320669,"[0.44676692937968154, 0.40209023644171343, 0.3...",20251210_103927_001N20ER30_MC_LR_SV_opt_10.json,TQATrainer,SV,0.960946
72421,erdos_renyi,20,20,LR_SV_opt.json,LR_opt.json,10,0.4,NaN,NaN,NaN,NaN,001_20nodes_erdosrenyi40percent.json,12.766411,"[0.4534196209286561, 0.4080776588357905, 0.362...",20251210_104033_001N20ER40_MC_LR_SV_opt_10.json,TQATrainer,SV,0.947431


## Get formatted and pivoted table and styler

With Pandas, dataframes are formatted with Stylers.
`qaoa_parameter_setting.utils.summary_table.formatted_styler_for` automatically
stylises the Styler and returns the pivoted dataframe and Styler.

### Example summary table with `"text"` formatting

The following cell formats the summary results into a single table suitable for
Jupyter notebooks. Later cells will format the table for LaTeX and save them to
a file.

#### MAXCUT Approximation Ratio

In [9]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values="approximation_ratio",
    with_fancy_values=False,
    cmap="Greens",
    precision=2,
    missing_data_str="-",
    target_format="text",
    acronym_mapping=ACRONYM_MAPPING,
    show_empty_rows=True,
    exclude_methods=["RTS"],
)
styler

In [10]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values="approximation_ratio",
    with_fancy_values=True,
    cmap={"MPS": "YlOrBr", "PP": "YlGn", "SV": "PuBu"},
    precision=2,
    missing_data_str="-",
    target_format="text",
    acronym_mapping=ACRONYM_MAPPING,
    show_empty_rows=True,
    exclude_methods=["RTS", "TS"],
)
styler

#### Number of instances and nodes

In [11]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values="num_instances",
    with_fancy_values=True,
    cmap="Greens",
    precision=0,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=False,
)
styler

## Combined table

In [12]:
table_cmap = {
    (k1, k2): v if k2 == "approximation_ratio" else "RdPu"
    for k1, v in {"MPS": "YlOrBr", "PP": "YlGn", "SV": "PuBu"}.items()
    for k2 in ["approximation_ratio", "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=5,
    agg_values=["approximation_ratio", "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision=2,
    missing_data_str="-",
    target_format="text",
    acronym_mapping=ACRONYM_MAPPING,
    show_empty_rows=False,
    exclude_methods=["RTS"],
)
styler

## Create and save tables for LaTeX

Here we format tables into LaTeX files and save them into separate `.tex` files
suitable for inclusion in a paper. Files are saved with the current date and
time for tracking changes. These can be compiled into a preview of all tables
with `pdflatex summary_tables.tex`

In [13]:
from qaoa_parameter_setting.utils.summary.summary_table_formatter import (
    convert_evaluation_to_multicolumn_latex,
)
from typing import TypeAlias

In [14]:
now = datetime.now(tz=timezone.utc)
generated_on_str = "% Generated on {now:%Y-%m-%d} at {now:%H:%M:%S} UTC\n".format(
    now=now
)
AggValue: TypeAlias = Literal["num_instances", "approximation_ratio", "both"]
# Open a summary list_of_tables file for previewing all tables.
with open("list_of_tables.tex", "w") as f:
    # Write date and time to list_of_tables
    f.write(generated_on_str)

    # Iterate over all depths, sorted so they're included in increasing order in
    # list_of_tables
    for depth in sorted(int(x) for x in table.to_dataframe()["depth"].unique()):
        # Write a section title.
        _ = f.write("\n\\section{{Tables for depth $P={}$}}\n".format(depth))

        # For each value to be plotted
        values_to_plot: list[tuple[AggValue, AggValue | list[AggValue], str, str]] = [
            (
                "num_instances",
                "num_instances",
                "Number of Instances (num. nodes in brackets)",
                " Cells are colored based on the number of instances for the given configuration, with darker colors indicating more instances.",
            ),
            (
                "approximation_ratio",
                "approximation_ratio",
                r"Avg. Approx. Ratio $\pm$ standard deviation in percentage",
                " Evaluation methods are shown in different colors, with darker colors indicating better approximation ratios.",
            ),
            (
                "both",
                ["approximation_ratio", "num_instances"],
                "Avg. Approx. Ratio and Number of Instances for Various Evaluation Methods",
                " Evaluation methods are shown in different colors for approximation ratios."
                + " Darker colors indicating better approximation ratios or more instances."
                + " Approximation ratios are in percentage, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
        for filename_suffix, values, value_label, additional_caption in values_to_plot:
            # This is the filename for this table.
            _latex_filename = "table_p{:02}_{}.tex".format(int(depth), filename_suffix)

            # Get the styler
            _, styler, _ = formatted_styler_for(
                table,
                depths=depth,
                agg_values=values,
                with_fancy_values=True,
                cmap=(
                    "Greens"
                    if values == "num_instances"
                    else {"MPS": "YlOrBr", "PP": "YlGn", "SV": "PuBu"}
                )
                if isinstance(values, str)
                else {
                    (k1, k2): v if k2 == "approximation_ratio" else "RdPu"
                    for k1, v in {"MPS": "YlOrBr", "PP": "YlGn", "SV": "PuBu"}.items()
                    for k2 in ["approximation_ratio", "num_instances"]
                },
                precision={"approximation_ratio": 1, "num_instances": 0},
                missing_data_str="-",
                target_format="latex",
                acronym_mapping=ACRONYM_MAPPING,
                show_empty_rows=True,
            )

            # Save table to separate LaTeX file
            with open(_latex_filename, "w") as f_table:
                _ = f_table.write(generated_on_str)
                _ = f_table.write(
                    "% Data is {value_label} for depth P={depth}.\n".format(
                        value_label=value_label, depth=depth
                    )
                )
                f_table.write(
                    convert_evaluation_to_multicolumn_latex(
                        styler.to_latex(
                            # f_table,
                            convert_css=True,
                            hrules=True,
                            clines="skip-last;data",
                        )
                    )
                )
            _ = f.write(
                r"""
\begin{{table}}[H]
    \centering
    \input{{{filename}}}
    \caption{{\textbf{{{value_label} for $P={depth}$.}} Graph types are Erdos Renyi (ER), Heavy-Hex (HH), Line-to-Full (L2F), and Random Regular (RR).{additional_caption}}}
\end{{table}}
""".format(
                    filename=_latex_filename,
                    depth=depth,
                    value_label=value_label,
                    additional_caption=additional_caption,
                )
            )
        _ = f.write(r"\clearpage")

## Compiling and previewing all tables

Tables are written to `table_p<depth>_<metric>.tex` where `<depth>` is the QAOA
depth and `<metric>` is either `approximation_ratio` or `num_instances`. These
LaTeX files contain the `tabular` environments to include the given tables.
Numerical values are formatted with siunitx for easier precision and uncertainty
handling. All of these tables are then included in `tables.tex` as a list of
sections, one per depth. `summary_tables.tex` is an example LaTeX document that
(i) has an appropriate preamble for rendering the tables and (ii) shows how to
include them in a larger document.

To compile the _sample document_ `summary_tables.tex`, run the following command
after generating the LaTeX files:

```bash
latexmk -pdf summary_tables.tex
```

In [15]:
!latexmk -pdf summary_tables.tex

Rc files read:
  NONE
Latexmk: This is Latexmk, John Collins, 15 June 2025. Version 4.87.
Latexmk: applying rule 'pdflatex'...
Rule 'pdflatex':  Reasons for rerun
Changed files or newly in use/created:
  list_of_tables.tex
  table_p01_approximation_ratio.tex
  table_p01_both.tex
  table_p01_num_instances.tex
  table_p02_approximation_ratio.tex
  table_p02_both.tex
  table_p02_num_instances.tex
  table_p03_approximation_ratio.tex
  table_p03_both.tex
  table_p03_num_instances.tex
  table_p04_approximation_ratio.tex
  table_p04_both.tex
  table_p04_num_instances.tex
  table_p05_approximation_ratio.tex
  table_p05_both.tex
  table_p05_num_instances.tex
  table_p06_approximation_ratio.tex
  table_p06_both.tex
  table_p06_num_instances.tex
  table_p07_approximation_ratio.tex
  table_p07_both.tex
  table_p07_num_instances.tex
  table_p08_approximation_ratio.tex
  table_p08_both.tex
  table_p08_num_instances.tex
  table_p09_approximation_ratio.tex
  table_p09_both.tex
  table_p09_num_instance